## Project: Text Classification with RNN
### Goal

- Build a model that classifies movie reviews as positive or negative.  

- You’ll load a dataset  

- Preprocess text  

- Build an RNN (LSTM)  

- Train & evaluate  

- Test sample sentences  

## Dataset  
We’ll use a small built-in dataset (so no downloads).

nltk comes with a sample movie review corpus — perfect for hands-on NLP.

## 0) Install dependencies

In [ ]:
!pip install jupyter lab

In [1]:
!pip install torch torchvision nltk numpy

  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/111.0 MB ? eta -:--:--
   - -------------------------------------- 3.4/111.0 MB 18.3 MB/s eta 0:00:06
   -- ------------------------------------- 8.1/111.0 MB 20.1 MB/s eta 0:00:06
   ---- ----------------------------------- 13.4/111.0 MB 21.5 MB/s eta 0:00:05
   ------ --------------------------------- 17.3/111.0 MB 21.0 MB/s eta 0:00:05
   ------- -------------------------------- 21.2/111.0 MB 20.3 MB/s eta 0:00:05
   --------- ------------------------------ 25.4/111.0 MB 20.1 MB/s eta 0:00:05
   ---------- ----------------------------- 29.4/111.0 MB 20.0 MB/s eta 0:00:05
   ------------ --------------------------- 34.1/111.0 MB 20.4 MB/s eta 0:00:04
   ------------- -------------------------- 38.0/111.0 MB 20.3 MB/s eta 0:00:04
   --------------- ------------------------ 41.9/111.0 MB 20.2 MB/s eta 0:00:04
   

## 1) Load sample dataset from NLTK

In [36]:
import nltk

# [단계 1] 데이터 준비: AI가 공부할 '교과서' 준비하기
# 컴퓨터가 글자를 배울 수 있도록 영화 리뷰 데이터를 불러옵니다.
nltk.download('movie_reviews')

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\dbwnd\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


True

## 2) Prepare Data

### 2-1)

In [68]:
# nltk라는 거대한 도구 상자에서 '영화 리뷰' 전용 서랍(데이터셋)을 가져옵니다.
from nltk.corpus import movie_reviews
# 리스트의 순서를 무작위로 섞어주는 '섞기 장인' 도구를 가져옵니다.
import random

# 1. 데이터를 담을 빈 바구니(리스트)를 만듭니다.
data = []

# 2. movie_reviews 서랍에 들어있는 파일 이름(ID)들을 하나씩 꺼내서 반복합니다.
for fileid in movie_reviews.fileids():
    # 해당 파일 이름(fileid)을 열어서 그 안에 적힌 단어(words)들을 몽땅 읽어옵니다.
    # 예: ['this', 'movie', 'is', 'good', ...]
    text = movie_reviews.words(fileid)
    
    # 이 리뷰가 '긍정(pos)'인지 '부정(neg)'인지 적힌 이름표(카테고리)를 가져옵니다.
    # [0]을 붙이는 이유는 목록 형태로 가져오기 때문에 첫 번째 진짜 이름표만 쏙 빼기 위함입니다.
    label = movie_reviews.categories(fileid)[0]
    
    # 컴퓨터가 읽기 편하게 단어들 사이에 공백(" ")을 넣어 하나의 긴 문장으로 합칩니다.
    # " ".join(['a', 'b']) -> "a b" 가 됩니다.
    # 그 문장과 이름표를 짝지어서(튜플) 바구니(data)에 넣습니다.
    data.append((" ".join(text), label))

# 3. 데이터 섞기 (중요!)
# 데이터가 처음에는 긍정끼리, 부정끼리 모여있을 수 있습니다. 
# AI가 순서대로 공부하면 편견이 생길 수 있으므로, 카드를 섞듯 무작위로 섞어줍니다.
random.shuffle(data)

# 4. 공부용과 시험용으로 나누기
# 전체 2,000개의 데이터 중에서:
# 처음부터 1,500개까지는 AI가 보고 공부할 '교과서'로 사용합니다.
train_data = data[:1500]
# 1,501번째부터 끝까지(500개)는 공부한 내용을 확인하는 '시험지'로 남겨둡니다.
test_data  = data[1500:]

# 5. 잘 나뉘었는지 개수를 확인해봅니다.
print("공부용 데이터 개수:", len(train_data)) # 결과: 1500
print("시험용 데이터 개수:", len(test_data))  # 결과: 500

공부용 데이터 개수: 1500
시험용 데이터 개수: 500


### 2-2) 긍/부정 비율 고려하기
##### 
- 위(2-1)의 방법은 운이 나쁘면 1,500개 안에 '부정' 리뷰가 1,200개, '긍정' 리뷰가 300개만 들어갈 수도 있습니다.  
- word.isalpha() 사용 여부

In [76]:
# 긍정 리뷰와 부정 리뷰를 각각 똑같은 개수만큼 가져와서 '편견'을 없애줍니다.
pos_fileids = movie_reviews.fileids('pos')
neg_fileids = movie_reviews.fileids('neg')

data = []
# 긍정 1000개, 부정 1000개를 섞어서 데이터 리스트를 만듭니다.
for fileid in pos_fileids:
    # 단어들만 추출하고 소문자로 통일합니다.
    words = [word.lower() for word in movie_reviews.words(fileid) if word.isalpha()]
    data.append((" ".join(words), "pos"))
for fileid in neg_fileids:
    words = [word.lower() for word in movie_reviews.words(fileid) if word.isalpha()]
    data.append((" ".join(words), "neg"))

# 데이터를 무작위로 섞습니다. (순서대로 있으면 AI가 답을 외워버려요!)
random.shuffle(data)
train_data = data[:1500]
# 1,501번째부터 끝까지(500개)는 공부한 내용을 확인하는 '시험지'로 남겨둡니다.
test_data  = data[1500:]

# 5. 잘 나뉘었는지 개수를 확인해봅니다.
print("공부용 데이터 개수:", len(train_data)) 
print("시험용 데이터 개수:", len(test_data))  


공부용 데이터 개수: 1500
시험용 데이터 개수: 500


In [3]:
movie_reviews.fileids()

['neg/cv000_29416.txt',
 'neg/cv001_19502.txt',
 'neg/cv002_17424.txt',
 'neg/cv003_12683.txt',
 'neg/cv004_12641.txt',
 'neg/cv005_29357.txt',
 'neg/cv006_17022.txt',
 'neg/cv007_4992.txt',
 'neg/cv008_29326.txt',
 'neg/cv009_29417.txt',
 'neg/cv010_29063.txt',
 'neg/cv011_13044.txt',
 'neg/cv012_29411.txt',
 'neg/cv013_10494.txt',
 'neg/cv014_15600.txt',
 'neg/cv015_29356.txt',
 'neg/cv016_4348.txt',
 'neg/cv017_23487.txt',
 'neg/cv018_21672.txt',
 'neg/cv019_16117.txt',
 'neg/cv020_9234.txt',
 'neg/cv021_17313.txt',
 'neg/cv022_14227.txt',
 'neg/cv023_13847.txt',
 'neg/cv024_7033.txt',
 'neg/cv025_29825.txt',
 'neg/cv026_29229.txt',
 'neg/cv027_26270.txt',
 'neg/cv028_26964.txt',
 'neg/cv029_19943.txt',
 'neg/cv030_22893.txt',
 'neg/cv031_19540.txt',
 'neg/cv032_23718.txt',
 'neg/cv033_25680.txt',
 'neg/cv034_29446.txt',
 'neg/cv035_3343.txt',
 'neg/cv036_18385.txt',
 'neg/cv037_19798.txt',
 'neg/cv038_9781.txt',
 'neg/cv039_5963.txt',
 'neg/cv040_8829.txt',
 'neg/cv041_22364.txt',


In [4]:
fileid

'pos/cv999_13106.txt'

In [ ]:
# text
movie_reviews.words(fileid)

['truman', '(', '"', 'true', '-', 'man', '"', ')', ...]

In [7]:
# label
movie_reviews.categories(fileid)[0]

'pos'

In [13]:
data

[('here i sit at my computer about to write my review of the recent action comedy " bait " starring jamie foxx and david morse . this is a review i don \' t even want to write because i \' d just be laying the same criticisms on it that i would with any other so - generic - and - predictable - it \' s - beyond - ridiculous piece of hollywood fluff . if they \' re not going to give us , the audience , just a little credit and put something together with half a brain , why should i waste my time and mental energy criticising it ? last summer i took this same approach with my review of " the mummy , " in that review i just quoted phrases my reviews of other sub - par movies . i think i shall do the same thing here but with a few less quotes ( not all are applicable ) . i hope this goes to show you what i think of " bait " and why you can find out all you need to know about it without having to take a wild guess . it \' s genuinely unfunny ( i , and the other audience members only laughed 

## 3) Tokenizer & Vocabulary

In [77]:
import torch
# Counter는 항목의 개수를 자동으로 세어주는 '똑똑한 계산기' 도구입니다.
from collections import Counter

# 1. 모든 단어를 한곳에 모으기 (단어 수집)
# train_data에 들어있는 모든 문장을 하나씩 꺼내서 단어 단위로 쪼갠 뒤(split), 
# 모두 소문자(lower)로 바꿔서 아주 긴 단어 리스트로 만듭니다.
# 비유: 마을에 있는 모든 책을 가져와서 단어 하나하나를 가위로 오려낸 것입니다.
all_tokens = [word.lower() for text, _ in train_data for word in text.split()]

# 2. 단어 빈도수 계산하기
# Counter가 오려낸 단어들을 보며 "love는 100번 나왔네?", "bad는 50번 나왔네?"라고 횟수를 셉니다.
vocab = Counter(all_tokens)

# 3. 가장 많이 쓰이는 중요 단어 고르기 (N개 선정)
# 너무 드물게 나오는 단어는 AI가 배우기 어려우므로, 가장 많이 등장한 상위 5,000개만 고릅니다.
vocab_size = 10000
most_common = vocab.most_common(vocab_size)

# 4. 번호표 매기기 (사전 만들기)
# itos: Index To String (번호를 주면 단어가 나오는 리스트)
# 예: ['the', 'movie', 'is', ...] -> 0번은 'the', 1번은 'movie'...
itos = [w for w, _ in most_common]

# stoi: String To Index (단어를 주면 번호가 나오는 사전)
# enumerate(itos)는 단어들에 0, 1, 2... 번호를 붙여줍니다.
# i+1을 하는 이유는 '0번'을 문장 길이를 맞추는 '빈칸(Padding)'용으로 남겨두기 위해서입니다.
# 예: {'the': 1, 'movie': 2, 'is': 3, ...}
stoi = {w: i+1 for i, w in enumerate(itos)}

# 5. 완성된 단어장의 크기를 확인해봅니다.
print("완성된 단어 사전의 크기:", len(stoi))

완성된 단어 사전의 크기: 10000


In [47]:
all_tokens

['claire',
 'danes',
 'giovanni',
 'ribisi',
 'and',
 'omar',
 'epps',
 'make',
 'a',
 'likable',
 'trio',
 'of',
 'protagonists',
 'but',
 'they',
 're',
 'just',
 'about',
 'the',
 'only',
 'palatable',
 'element',
 'of',
 'the',
 'mod',
 'squad',
 'a',
 'lame',
 'brained',
 'big',
 'screen',
 'version',
 'of',
 'the',
 'tv',
 'show',
 'the',
 'story',
 'has',
 'all',
 'the',
 'originality',
 'of',
 'a',
 'block',
 'of',
 'wood',
 'well',
 'it',
 'would',
 'if',
 'you',
 'could',
 'decipher',
 'it',
 'the',
 'characters',
 'are',
 'all',
 'blank',
 'slates',
 'and',
 'scott',
 'silver',
 's',
 'perfunctory',
 'action',
 'sequences',
 'are',
 'as',
 'cliched',
 'as',
 'they',
 'come',
 'by',
 'sheer',
 'force',
 'of',
 'talent',
 'the',
 'three',
 'actors',
 'wring',
 'marginal',
 'enjoyment',
 'from',
 'the',
 'proceedings',
 'whenever',
 'they',
 're',
 'on',
 'screen',
 'but',
 'the',
 'mod',
 'squad',
 'is',
 'just',
 'a',
 'second',
 'rate',
 'action',
 'picture',
 'with',
 'a',


In [48]:
vocab

Counter({'the': 60644,
         'a': 30181,
         'and': 28238,
         'of': 26848,
         'to': 25383,
         'is': 20037,
         'in': 17262,
         's': 14514,
         'it': 12885,
         'that': 12769,
         'as': 8925,
         'with': 8535,
         'for': 7849,
         'this': 7652,
         'his': 7544,
         'film': 7503,
         'i': 7170,
         'he': 7055,
         'but': 6847,
         'on': 5820,
         'are': 5491,
         't': 5129,
         'by': 4909,
         'be': 4828,
         'one': 4665,
         'movie': 4630,
         'who': 4560,
         'an': 4542,
         'not': 4527,
         'you': 4272,
         'from': 4006,
         'have': 3978,
         'was': 3939,
         'at': 3934,
         'they': 3863,
         'has': 3749,
         'her': 3573,
         'all': 3481,
         'there': 2999,
         'like': 2998,
         'so': 2932,
         'out': 2911,
         'about': 2794,
         'up': 2740,
         'what': 2701,
       

In [49]:
most_common

[('the', 60644),
 ('a', 30181),
 ('and', 28238),
 ('of', 26848),
 ('to', 25383),
 ('is', 20037),
 ('in', 17262),
 ('s', 14514),
 ('it', 12885),
 ('that', 12769),
 ('as', 8925),
 ('with', 8535),
 ('for', 7849),
 ('this', 7652),
 ('his', 7544),
 ('film', 7503),
 ('i', 7170),
 ('he', 7055),
 ('but', 6847),
 ('on', 5820),
 ('are', 5491),
 ('t', 5129),
 ('by', 4909),
 ('be', 4828),
 ('one', 4665),
 ('movie', 4630),
 ('who', 4560),
 ('an', 4542),
 ('not', 4527),
 ('you', 4272),
 ('from', 4006),
 ('have', 3978),
 ('was', 3939),
 ('at', 3934),
 ('they', 3863),
 ('has', 3749),
 ('her', 3573),
 ('all', 3481),
 ('there', 2999),
 ('like', 2998),
 ('so', 2932),
 ('out', 2911),
 ('about', 2794),
 ('up', 2740),
 ('what', 2701),
 ('when', 2644),
 ('more', 2633),
 ('or', 2528),
 ('she', 2516),
 ('their', 2477),
 ('which', 2472),
 ('some', 2384),
 ('just', 2345),
 ('can', 2279),
 ('we', 2228),
 ('if', 2227),
 ('him', 2160),
 ('even', 2086),
 ('into', 2041),
 ('only', 2004),
 ('no', 1998),
 ('than', 1985

In [50]:
itos

['the',
 'a',
 'and',
 'of',
 'to',
 'is',
 'in',
 's',
 'it',
 'that',
 'as',
 'with',
 'for',
 'this',
 'his',
 'film',
 'i',
 'he',
 'but',
 'on',
 'are',
 't',
 'by',
 'be',
 'one',
 'movie',
 'who',
 'an',
 'not',
 'you',
 'from',
 'have',
 'was',
 'at',
 'they',
 'has',
 'her',
 'all',
 'there',
 'like',
 'so',
 'out',
 'about',
 'up',
 'what',
 'when',
 'more',
 'or',
 'she',
 'their',
 'which',
 'some',
 'just',
 'can',
 'we',
 'if',
 'him',
 'even',
 'into',
 'only',
 'no',
 'than',
 'good',
 'time',
 'most',
 'its',
 'will',
 'story',
 'would',
 'much',
 'been',
 'character',
 'do',
 'also',
 'get',
 'well',
 'other',
 'two',
 'them',
 'characters',
 'very',
 'see',
 'after',
 'first',
 'because',
 'way',
 'make',
 'really',
 'too',
 'does',
 'any',
 'off',
 'films',
 'life',
 'how',
 'plot',
 'while',
 'had',
 'where',
 'little',
 'people',
 'bad',
 'my',
 'over',
 'me',
 'could',
 'then',
 'man',
 'never',
 'being',
 'don',
 'scene',
 'best',
 'scenes',
 'doesn',
 'these',


In [51]:
stoi

{'the': 1,
 'a': 2,
 'and': 3,
 'of': 4,
 'to': 5,
 'is': 6,
 'in': 7,
 's': 8,
 'it': 9,
 'that': 10,
 'as': 11,
 'with': 12,
 'for': 13,
 'this': 14,
 'his': 15,
 'film': 16,
 'i': 17,
 'he': 18,
 'but': 19,
 'on': 20,
 'are': 21,
 't': 22,
 'by': 23,
 'be': 24,
 'one': 25,
 'movie': 26,
 'who': 27,
 'an': 28,
 'not': 29,
 'you': 30,
 'from': 31,
 'have': 32,
 'was': 33,
 'at': 34,
 'they': 35,
 'has': 36,
 'her': 37,
 'all': 38,
 'there': 39,
 'like': 40,
 'so': 41,
 'out': 42,
 'about': 43,
 'up': 44,
 'what': 45,
 'when': 46,
 'more': 47,
 'or': 48,
 'she': 49,
 'their': 50,
 'which': 51,
 'some': 52,
 'just': 53,
 'can': 54,
 'we': 55,
 'if': 56,
 'him': 57,
 'even': 58,
 'into': 59,
 'only': 60,
 'no': 61,
 'than': 62,
 'good': 63,
 'time': 64,
 'most': 65,
 'its': 66,
 'will': 67,
 'story': 68,
 'would': 69,
 'much': 70,
 'been': 71,
 'character': 72,
 'do': 73,
 'also': 74,
 'get': 75,
 'well': 76,
 'other': 77,
 'two': 78,
 'them': 79,
 'characters': 80,
 'very': 81,
 'see': 

## 4) Encode Examples & Pads

In [78]:
# 함수 정의: 문장(sent)을 받아서, 정해진 길이(max_len)의 숫자 묶음으로 바꿉니다.
def encode_sentence(sent, max_len=200):
    
    # 1. 소문자로 바꾸고 단어별로 자르기
    # "Good Movie" -> "good movie" -> ["good", "movie"]
    tokens = sent.lower().split()
    
    # 2. 단어를 번호로 바꾸기 (가장 중요한 부분!)
    # 아까 만든 단어 사전(stoi)에서 단어의 번호를 찾아옵니다.
    # 만약 사전에 없는 모르는 단어라면 0번(기본값)으로 가져옵니다.
    # [:max_len]은 문장이 너무 길면 뒤쪽은 과감히 잘라내겠다는 뜻입니다.
    encoded = [stoi.get(w, 0) for w in tokens[:max_len]]
    
    # 3. 길이 맞추기 (패딩, Padding)
    # 인공지능은 입력받는 상자의 크기가 모두 똑같아야 합니다.
    # 문장이 설정한 길이(200단어)보다 짧다면, 남는 뒷부분을 0으로 채워줍니다.
    # 예: [5, 12] -> [5, 12, 0, 0, 0, ...] (총 200개가 될 때까지)
    padded = encoded + [0] * (max_len - len(encoded))
    
    # 4. 파이토치 텐서로 변환
    # 파이썬 리스트를 인공지능 도구(PyTorch)가 계산할 수 있는 형태인 '텐서'로 바꿔서 반환합니다.
    return torch.tensor(padded)

## 5) Dataset + DataLoader

In [79]:
# 파이토치(torch)에서 데이터를 다루기 위한 기본 도구들을 가져옵니다.
from torch.utils.data import Dataset, DataLoader

# [1] MovieDataset 클래스: 데이터를 하나씩 꺼내기 좋게 정리하는 '정리함'입니다.
class MovieDataset(Dataset):
    # 처음 클래스를 만들 때 데이터를 받아서 저장해둡니다.
    def __init__(self, data):
        self.data = data

    # 전체 데이터가 몇 개인지 알려주는 기능입니다.
    def __len__(self):
        return len(self.data)

    # 데이터 상자에서 '몇 번째(idx)' 데이터를 꺼낼지 결정하는 핵심 기능입니다.
    def __getitem__(self, idx):
        # 1. 원본 데이터에서 텍스트와 라벨(긍정/부정)을 꺼냅니다.
        text, label = self.data[idx]
        
        # 2. 아까 만든 함수를 이용해 문장을 '숫자 묶음(번호표)'으로 바꿉니다.
        x = encode_sentence(text)
        
        # 3. 글자로 된 라벨을 숫자로 바꿉니다. (긍정은 1, 부정은 0)
        y = 1 if label == "pos" else 0
        
        # 4. 숫자 문장(x)과 정답 번호(y)를 쌍으로 반환합니다.
        return x, torch.tensor(y)

# [2] DataLoader: 인공지능이 공부할 때 데이터를 '한 입 크기'로 나누어 주는 역할입니다.
# batch_size=32: 데이터를 한 번에 32개씩 묶어서 전달합니다. (너무 많이 주면 인공지능이 체해요!)
# shuffle=True: 공부할 때 순서를 외우지 못하게 무작위로 섞어서 줍니다.
train_loader = DataLoader(MovieDataset(train_data), batch_size=32, shuffle=True)

# 시험용 데이터는 섞을 필요가 없으므로 shuffle을 하지 않습니다.
test_loader  = DataLoader(MovieDataset(test_data),  batch_size=32)

## 6) RNN Model (LSTM)

In [80]:
import torch.nn as nn

# SentimentRNN: 문장을 읽고 감정을 분석하는 인공지능 모델 클래스입니다.
class SentimentRNN(nn.Module):
    # [1단계: 부품 준비하기] 모델이 사용할 도구들을 정의합니다.
    def __init__(self, vocab_size, embed_dim, hidden_size):
        super().__init__()
        
        # 1. Embedding(임베딩): 번호표(숫자)를 의미를 가진 '공간상의 좌표'로 바꿉니다.
        # 비유: 사전에서 단어의 뜻을 찾아 '긍정-부정' 지도의 어느 위치인지 점을 찍는 것과 같습니다.
        # padding_idx=0: 0번(빈칸)은 아무 의미가 없으므로 계산에서 제외하라는 뜻입니다.
        self.embed = nn.Embedding(vocab_size + 1, embed_dim, padding_idx=0)
        
        # 2. LSTM: 문맥을 기억하며 읽는 '똑똑한 RNN'입니다.
        # 비유: 앞 단어의 내용을 잊지 않고 기억하면서 다음 단어를 읽어나가는 '기억력 장치'입니다.
        # batch_first=True: 데이터가 [묶음, 문장길이, 단어] 순서로 들어온다는 설정입니다.
        self.lstm = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        
        # 3. Linear(선형층): 마지막 기억을 가지고 결론을 내리는 '최종 판단관'입니다.
        # LSTM이 정리한 정보들을 모아서 하나의 점수(결과)를 만듭니다.
        self.fc = nn.Linear(hidden_size, 1)
        
        # 4. Sigmoid(시그모이드): 점수를 0과 1 사이의 '확률'로 바꿔줍니다.
        # 0.5보다 크면 긍정, 작으면 부정으로 해석하기 위함입니다.
        self.act = nn.Sigmoid()

    # [2단계: 실제로 실행하기] 데이터가 들어왔을 때 어떻게 처리할지 순서를 정합니다.
    def forward(self, x):
        # 1. 숫자로 된 문장(x)을 좌표(emb)로 바꿉니다.
        emb = self.embed(x)
        
        # 2. LSTM이 좌표들을 순서대로 읽으며 문장의 맥락을 파악합니다.
        out, _ = self.lstm(emb)
        
        # 3. 문장을 끝까지 다 읽었을 때의 '마지막 기억'(-1번째)만 쏙 뽑아냅니다.
        # 전체 문장을 읽고 난 후의 최종 느낌을 가져오는 것입니다.
        out = out[:, -1]
        
        # 4. 마지막 기억을 토대로 점수를 내고, 시그모이드를 통과시켜 최종 확률을 냅니다.
        out = self.fc(out)
        return self.act(out)

# 모델 생성: 단어장 크기, 좌표의 크기(128), 기억력의 크기(128)를 설정합니다.
model = SentimentRNN(vocab_size, embed_dim=128, hidden_size=128)

## 7) Train Loop

In [81]:
import torch.optim as optim

# 1. 채점 기준(Loss Function) 정하기
# BCELoss: '맞다(1)/틀리다(0)'를 구분하는 문제에서 정답과 얼마나 거리가 먼지 계산하는 채점지입니다.
# 틀릴수록 점수(Loss)가 높게 나옵니다.
criterion = nn.BCELoss()

# 2. 공부 방법(Optimizer) 정하기
# Adam: 가장 똑똑하게 오답을 수정하는 '공부 도우미'입니다.
# lr=1e-3 (0.001): 한 번 배울 때 얼마나 큰 폭으로 지식을 수정할지 결정하는 '학습 속도'입니다.
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 3. 본격적인 반복 학습 시작
for epoch in range(15): # 전체 문제집을 총 5번 반복해서 봅니다.
    total_loss = 0     # 이번 회차에 얼마나 많이 틀렸는지 합계를 낼 변수입니다.
    model.train()      # "이제부터 공부 시작이야!"라고 모델에게 알려줍니다. (공부 모드 활성화)

    # 배달부(train_loader)가 가져다주는 32개씩의 문제들을 하나씩 풉니다.
    for x, y in train_loader:
        # (1) 문제 풀기: AI가 숫자로 된 문장(x)을 보고 0~1 사이의 답을 냅니다.
        pred = model(x)
        
        # (2) 채점하기: AI가 낸 답(pred)과 실제 정답(y)을 비교해 '틀린 점수(loss)'를 냅니다.
        loss = criterion(pred.squeeze(), y.float())

        # (3) 오답 노트 준비: 이전 문제에서 계산했던 틀린 기록을 깨끗이 지웁니다.
        optimizer.zero_grad()
        
        # (4) 원인 파악: "왜 틀렸지?"를 분석하며 모델의 뇌 세포(파라미터)들을 거꾸로 추적합니다.
        loss.backward()
        
        # (5) 실력 수정: 분석한 내용을 바탕으로 모델의 머릿속 내용을 조금 수정합니다.
        optimizer.step()

        # 틀린 점수를 계속 누적합니다.
        total_loss += loss.item()
    
    # 한 회차(Epoch)가 끝날 때마다 평균적으로 얼마나 틀렸는지(Loss) 출력합니다.
    # 공부를 잘하고 있다면 이 숫자가 점점 줄어들어야 합니다!
    print(f"Epoch {epoch+1}, loss {total_loss/len(train_loader):.4f}")

Epoch 1, loss 0.6936
Epoch 2, loss 0.6556
Epoch 3, loss 0.5754
Epoch 4, loss 0.4454
Epoch 5, loss 0.2850
Epoch 6, loss 0.1471
Epoch 7, loss 0.0711
Epoch 8, loss 0.0469
Epoch 9, loss 0.0201
Epoch 10, loss 0.0165
Epoch 11, loss 0.0417
Epoch 12, loss 0.1787
Epoch 13, loss 0.3024
Epoch 14, loss 0.1086
Epoch 15, loss 0.0561


## 8) Evaluate

In [82]:
# 1. 평가 모드 전환
# "이제 공부 끝! 실전 시험이야. 답 고치지 말고 있는 실력 그대로 보여줘."
# 모델에게 더 이상 학습(가중치 수정)을 하지 말라고 명령하는 것입니다.
model.eval()

# 2. 점수 계산을 위한 변수 준비
correct = 0  # 맞힌 문제 수를 셀 바구니
total   = 0  # 전체 문제 수를 셀 바구니

# 3. 기울기 계산 비활성화 (메모리 절약)
# "시험 볼 때는 오답 노트를 쓸 필요가 없으니, 복잡한 계산은 하지 마."
# 예측만 하면 되기 때문에 계산 속도를 높이고 메모리를 아끼기 위해 사용합니다.
with torch.no_grad():
    # 시험지 배달부(test_loader)가 가져오는 문제들을 하나씩 풉니다.
    for x, y in test_loader:
        # (1) 예측하기: 공부한 뇌(model)를 이용해 결과를 냅니다.
        pred = model(x)
        
        # (2) 결과 판정: 0.5보다 크면 1(긍정), 작으면 0(부정)으로 딱 잘라 답을 냅니다.
        # .long()은 결과를 정수(0 또는 1) 형태로 바꾸라는 뜻입니다.
        predicted = (pred.squeeze() > 0.5).long()
        
        # (3) 채점하기: AI의 답(predicted)과 실제 정답(y)이 일치하는 개수를 셉니다.
        correct += (predicted == y).sum().item()
        
        # (4) 전체 문제 수 누적: 푼 문제의 총 개수를 더해갑니다.
        total += y.size(0)

# 4. 최종 정확도 출력
# (맞힌 개수 / 전체 문제 수)를 계산하여 0~1 사이의 값으로 보여줍니다.
# 예: 0.85가 나오면 "85점을 맞혔구나!"라고 이해하면 됩니다.
print("최종 시험 정확도(Accuracy):", correct/total)

최종 시험 정확도(Accuracy): 0.55


## 9) Try Custom Sentence

In [83]:
# predict_sent: 우리가 평소에 쓰는 문장(sent)을 넣으면 '긍정'인지 '부정'인지 알려주는 함수입니다.
def predict_sent(sent):
    # 1. 문장을 바코드로 바꾸기
    # 앞에서 만든 encode_sentence 함수를 사용해 문장을 숫자 묶음으로 바꿉니다.
    # .unsqueeze(0): AI는 항상 '묶음(Batch)' 단위로 데이터를 받습니다. 
    # 비록 문장이 1개지만, 1개짜리 묶음이라고 표시해주는 과정입니다.
    x = encode_sentence(sent).unsqueeze(0)  
    
    # 2. 결과 예측하기 (시험 모드)
    # with torch.no_grad(): 평가할 때는 공부(수정)하지 말고 답만 내라는 뜻입니다.
    with torch.no_grad():
        # model(x)에 숫자로 바뀐 문장을 넣으면 0~1 사이의 확률값(pred)이 나옵니다.
        # .item()은 텐서 형태의 결과에서 숫자만 쏙 꺼내오는 기능입니다.
        pred = model(x).item()
    
    # 3. 사람이 읽을 수 있게 번역하기
    # 확률이 0.5(50%)보다 높으면 "Positive(긍정)", 낮으면 "Negative(부정)"라고 대답합니다.
    return "Positive" if pred > 0.5 else "Negative"

# --- 직접 만든 문장으로 테스트해보기 ---
# 긍정적인 문장을 넣었을 때 AI가 "Positive"라고 답하는지 확인합니다.
print(predict_sent("I loved this movie!"))

# 부정적인 문장을 넣었을 때 AI가 "Negative"라고 답하는지 확인합니다.
print(predict_sent("This was the worst film ever"))

Positive
Negative


# 성능을 높이기 위해선?  
- 학습 데이터(긍/부정 비율, 학습/테스트 비율, 학습데이터 양)
- 어휘집(vocab) 사이즈
- 학습 수(epoch 수)
